# 📖 Notebook 4: Explore & Recommendation

Instagram's **Explore page** shows you posts from accounts you don't follow —  
content you'll probably enjoy based on what you've liked, saved, and engaged with.

Users spend over **50% of their time** on Explore, making it one of Instagram's  
most important features for discovery and engagement.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **collaborative filtering** works ("users like you also liked...")
- **Engagement scoring** — ranking posts by predicted interest
- The difference between **candidate generation** and **ranking**
- How to cache recommendations for low-latency serving
- Trade-offs between freshness and pre-computation

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/instagram
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json
from collections import defaultdict

# ── Connections ───────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM user_interactions")
    print(f"✅ PostgreSQL — {cur.fetchone()[0]} user interactions")
    cur.execute("SELECT COUNT(*) FROM likes")
    print(f"   {cur.fetchone()[0]} likes")
    cur.execute("SELECT COUNT(*) FROM posts")
    print(f"   {cur.fetchone()[0]} posts")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Explore Problem

The feed shows posts from people you **already follow**.  
Explore shows posts from people you **don't follow but might like**.

How do we decide what to show? Two main signals:

1. **What you've engaged with** — posts you liked, saved, commented on, or spent time viewing
2. **What similar users like** — if users with similar tastes liked something, you might too

The process has two stages:

```
Stage 1: CANDIDATE GENERATION          Stage 2: RANKING
"Find 1000 posts you MIGHT like"       "Pick the best 50 from those 1000"

┌──────────────┐                       ┌──────────────┐
│ Collaborative│                       │              │
│ Filtering    │──┐                    │  Engagement  │
│              │  │                    │  Scoring     │
├──────────────┤  ├──► 1000 posts ──►  │  Model       │──► Top 50 posts
│ Content      │  │    candidates      │              │
│ Similarity   │──┘                    │              │
│              │                       └──────────────┘
└──────────────┘
```

Let's build each stage.

## 🤝 Stage 1: Collaborative Filtering

**Collaborative filtering** is the simplest recommendation approach:  
"Users who liked the same posts as you also liked these other posts."

The algorithm:
1. Find posts that User A has liked
2. Find other users who liked the same posts ("similar users")
3. Find posts those similar users liked that User A hasn't seen
4. Rank by how many similar users liked each post

```
You liked posts: [10, 25, 42]

User 7 also liked posts [10, 25, 99, 150]      ← similar to you!
User 12 also liked posts [25, 42, 88, 200]      ← similar to you!
User 30 liked posts [300, 400]                   ← NOT similar

Candidates: posts [99, 150, 88, 200] (liked by similar users, not by you)
```

In [ ]:
def find_similar_users(user_id: int, min_overlap: int = 2) -> list:
    """
    Find users who have liked many of the same posts as this user.
    Returns users sorted by overlap count (most similar first).
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        -- Step 1: Get posts that this user liked
        WITH my_likes AS (
            SELECT post_id FROM likes WHERE user_id = %s
        ),
        -- Step 2: Find other users who liked the same posts
        similar AS (
            SELECT l.user_id AS similar_user_id,
                   COUNT(*) AS overlap_count
            FROM likes l
            JOIN my_likes ml ON ml.post_id = l.post_id
            WHERE l.user_id != %s
            GROUP BY l.user_id
            HAVING COUNT(*) >= %s
            ORDER BY COUNT(*) DESC
            LIMIT 20
        )
        SELECT s.similar_user_id, s.overlap_count,
               u.username, u.display_name
        FROM similar s
        JOIN users u ON u.id = s.similar_user_id
        ORDER BY s.overlap_count DESC
    """, (user_id, user_id, min_overlap))

    results = cur.fetchall()
    conn.close()
    return results

# Find users similar to User 1
print("🔍 Finding users similar to User 1...\n")
similar = find_similar_users(user_id=1, min_overlap=1)

if similar:
    print(f"Found {len(similar)} similar users:")
    print(f"{'User':<20} {'Overlap':>10}")
    print("-" * 32)
    for s in similar[:10]:
        print(f"{s['display_name']:<20} {s['overlap_count']:>10} shared likes")
else:
    print("No similar users found (need more interaction data).")
    print("This is expected with our small demo dataset.")

In [ ]:
def get_collaborative_candidates(user_id: int, limit: int = 50) -> list:
    """
    Collaborative filtering: find posts liked by similar users
    that this user hasn't seen yet and doesn't follow the author.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        -- Posts liked by similar users that I haven't liked
        -- and that aren't from people I already follow
        WITH my_likes AS (
            SELECT post_id FROM likes WHERE user_id = %s
        ),
        my_follows AS (
            SELECT followee_id FROM follows WHERE follower_id = %s
        ),
        similar_users AS (
            SELECT l.user_id AS similar_user_id, COUNT(*) AS overlap
            FROM likes l
            JOIN my_likes ml ON ml.post_id = l.post_id
            WHERE l.user_id != %s
            GROUP BY l.user_id
            HAVING COUNT(*) >= 1
            ORDER BY COUNT(*) DESC
            LIMIT 20
        )
        SELECT p.id AS post_id, p.author_id, p.caption, p.like_count,
               p.comment_count, p.created_at,
               COUNT(DISTINCT su.similar_user_id) AS recommended_by_count
        FROM posts p
        JOIN likes l ON l.post_id = p.id
        JOIN similar_users su ON su.similar_user_id = l.user_id
        WHERE p.id NOT IN (SELECT post_id FROM my_likes)
          AND p.author_id NOT IN (SELECT followee_id FROM my_follows)
          AND p.author_id != %s
        GROUP BY p.id
        ORDER BY COUNT(DISTINCT su.similar_user_id) DESC, p.like_count DESC
        LIMIT %s
    """, (user_id, user_id, user_id, user_id, limit))

    results = cur.fetchall()
    conn.close()
    return results

print("🔍 Finding explore candidates for User 1 (collaborative filtering)...\n")
candidates = get_collaborative_candidates(user_id=1)

if candidates:
    print(f"Found {len(candidates)} candidate posts:")
    print(f"{'Post ID':<10} {'Author':<10} {'Likes':>8} {'Reco By':>10} {'Caption':<30}")
    print("-" * 72)
    for c in candidates[:10]:
        print(f"{c['post_id']:<10} User {c['author_id']:<5} {c['like_count']:>8} {c['recommended_by_count']:>10} {c['caption'][:30]}")
else:
    print("No candidates found — this is normal with our small dataset.")
    print("In production, Instagram has billions of interactions to work with.")

## 📈 Stage 2: Engagement Scoring

Collaborative filtering gives us **candidates** — posts the user might like.  
But we need to **rank** them: which candidate should appear first?

Instagram uses machine learning for this, but we can build a simple scoring model  
that captures the key ideas:

```
score = (engagement_rate × 0.4)
      + (recency_score   × 0.3)
      + (author_quality  × 0.2)
      + (diversity_bonus × 0.1)
```

| Signal | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Engagement rate** | likes / views ratio | High engagement = interesting content |
| **Recency** | How recent the post is | Fresh content is more relevant |
| **Author quality** | Author's average engagement | Consistently good creators |
| **Diversity** | Spread across different authors | Avoid showing 10 posts from 1 person |

In [ ]:
import math

def score_post(post: dict, seen_authors: set) -> float:
    """
    Calculate a score for a post to determine its ranking in the Explore feed.
    Higher score = shown higher in the feed.
    """
    # ── Engagement rate (0 to 1) ─────────────────────────────
    # Normalize like_count using log scale (prevents viral posts from dominating)
    engagement = math.log(max(post["like_count"], 1) + 1) / 12.0
    engagement = min(engagement, 1.0)

    # ── Recency score (0 to 1) ───────────────────────────────
    # Posts from the last 24h get score 1.0, older posts decay
    age_hours = (time.time() - post["created_at"].timestamp()) / 3600
    recency = max(0, 1.0 - (age_hours / 168))  # decay over 7 days

    # ── Author quality (0 to 1) ──────────────────────────────
    avg_likes = post.get("author_avg_likes", post["like_count"])
    author_quality = math.log(max(avg_likes, 1) + 1) / 12.0
    author_quality = min(author_quality, 1.0)

    # ── Diversity bonus ──────────────────────────────────────
    # Penalize posts from authors we've already shown
    diversity = 0.0 if post["author_id"] in seen_authors else 1.0

    # ── Weighted score ───────────────────────────────────────
    score = (
        engagement    * 0.4 +
        recency       * 0.3 +
        author_quality * 0.2 +
        diversity     * 0.1
    )

    return round(score, 4)

# Score our candidates
if candidates:
    print("📊 Scoring candidates...\n")
    seen_authors = set()
    scored = []

    for c in candidates:
        s = score_post(c, seen_authors)
        scored.append((s, c))
        seen_authors.add(c["author_id"])

    scored.sort(key=lambda x: x[0], reverse=True)

    print(f"{'Score':>7} {'Post':>6} {'Author':>8} {'Likes':>8} {'Caption':<35}")
    print("-" * 70)
    for score, post in scored[:10]:
        print(f"{score:>7.4f} #{post['post_id']:<5} User {post['author_id']:<3} {post['like_count']:>8} {post['caption'][:35]}")
else:
    print("No candidates to score. Using popularity-based fallback instead...")

## 🔥 Fallback: Popularity-Based Recommendations

Collaborative filtering needs enough user interactions to work.  
For **new users** (the "cold start" problem) or when we don't have enough data,  
we fall back to **popularity-based** recommendations.

This simply shows the most-liked posts from the past 24–48 hours  
that the user hasn't already seen.

In [ ]:
def get_popular_posts(user_id: int, hours: int = 48, limit: int = 30) -> list:
    """
    Popularity-based recommendations (cold start fallback).
    Returns trending posts from the last N hours that the user
    hasn't already seen (not in their feed, not from followed users).
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT p.id AS post_id, p.author_id, p.caption,
               p.like_count, p.comment_count, p.created_at,
               u.username, u.display_name
        FROM posts p
        JOIN users u ON u.id = p.author_id
        WHERE p.created_at > NOW() - interval '%s hours'
          AND p.author_id NOT IN (
              SELECT followee_id FROM follows WHERE follower_id = %s
          )
          AND p.author_id != %s
          AND p.media_upload_status = 'complete'
        ORDER BY p.like_count DESC
        LIMIT %s
    """, (hours, user_id, user_id, limit))

    results = cur.fetchall()
    conn.close()
    return results

print("🔥 Popular posts (fallback for cold start):\n")
popular = get_popular_posts(user_id=1, hours=720)  # wider window for demo

if popular:
    print(f"{'Rank':<6} {'Likes':>8} {'Author':<20} {'Caption':<35}")
    print("-" * 72)
    for i, p in enumerate(popular[:10], 1):
        print(f"{i:<6} {p['like_count']:>8} {p['display_name']:<20} {p['caption'][:35]}")
else:
    print("No popular posts found in the time window.")

## 💾 Caching Explore Results

Computing recommendations is expensive (lots of SQL joins, scoring, etc).  
We can't do this on every Explore page load.

Instead, we **pre-compute** recommendations and cache them in Redis:

```
Background job (runs every 15-30 minutes):
  For each active user:
    1. Generate candidates (collaborative filtering)
    2. Score and rank them
    3. Store top 200 in Redis: explore:{user_id}

When user opens Explore:
  1. Read from Redis (instant!)
  2. Filter out posts they've already seen
  3. Return top 30
```

In [ ]:
def precompute_explore(user_id: int):
    """
    Pre-compute and cache Explore recommendations for a user.
    In production, this runs as a batch job every 15-30 minutes.
    """
    r = get_redis()
    start = time.time()

    # Try collaborative filtering first
    candidates = get_collaborative_candidates(user_id, limit=100)

    # Fall back to popularity if not enough candidates
    if len(candidates) < 20:
        popular = get_popular_posts(user_id, hours=720, limit=100)
        # Convert to same format
        for p in popular:
            if not any(c["post_id"] == p["post_id"] for c in candidates):
                candidates.append(p)

    # Score all candidates
    seen_authors = set()
    scored = []
    for c in candidates:
        s = score_post(c, seen_authors)
        scored.append((s, c["post_id"]))
        seen_authors.add(c["author_id"])

    scored.sort(key=lambda x: x[0], reverse=True)

    # Store in Redis sorted set (score = recommendation score)
    explore_key = f"explore:{user_id}"
    r.delete(explore_key)
    if scored:
        pipeline = r.pipeline()
        for score, post_id in scored[:200]:  # keep top 200
            pipeline.zadd(explore_key, {str(post_id): score})
        pipeline.expire(explore_key, 1800)  # TTL: 30 minutes
        pipeline.execute()

    elapsed = (time.time() - start) * 1000
    print(f"✅ Pre-computed Explore for user {user_id}")
    print(f"   {len(scored)} posts cached in Redis (TTL: 30min)")
    print(f"   Computation time: {elapsed:.0f}ms")

def get_explore_feed(user_id: int, limit: int = 30) -> list:
    """
    Read pre-computed Explore feed from Redis.
    This is what happens when the user taps the Explore tab.
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # Read top N post IDs from Redis (sorted by score, highest first)
    explore_key = f"explore:{user_id}"
    post_ids_with_scores = r.zrevrange(explore_key, 0, limit - 1, withscores=True)

    if not post_ids_with_scores:
        print(f"Cache miss for user {user_id} — would trigger pre-computation")
        conn.close()
        return []

    # Hydrate post data
    post_ids = [int(pid) for pid, _ in post_ids_with_scores]
    scores = {int(pid): score for pid, score in post_ids_with_scores}

    cur.execute("""
        SELECT p.id AS post_id, p.author_id, p.caption, p.like_count,
               p.created_at, u.username, u.display_name
        FROM posts p
        JOIN users u ON u.id = p.author_id
        WHERE p.id = ANY(%s)
    """, (post_ids,))

    posts = cur.fetchall()
    # Add scores and sort by score
    for p in posts:
        p["explore_score"] = scores.get(p["post_id"], 0)
    posts.sort(key=lambda p: p["explore_score"], reverse=True)

    elapsed = (time.time() - start) * 1000
    conn.close()

    print(f"⚡ Explore feed for user {user_id}: {elapsed:.1f}ms")
    return posts

# Pre-compute and then read
print("Step 1: Pre-compute (background job)\n")
precompute_explore(user_id=1)

print(f"\nStep 2: Read Explore feed (user opens app)\n")
explore = get_explore_feed(user_id=1)

if explore:
    print(f"\n{'Score':>7} {'Likes':>7} {'Author':<20} {'Caption':<35}")
    print("-" * 72)
    for p in explore[:10]:
        print(f"{p['explore_score']:>7.4f} {p['like_count']:>7} {p['display_name']:<20} {p['caption'][:35]}")

## 📊 Explore Architecture Summary

```
                                    ┌──────────────────────────────────────┐
                                    │     BATCH JOB (every 15-30 min)     │
                                    │                                      │
                                    │  1. For each active user:            │
┌────────────┐                      │     - Collaborative filtering (SQL)  │
│ PostgreSQL │◄─────────────────────│     - Score & rank candidates        │
│            │  read interactions   │     - Store top 200 in Redis         │
│ likes      │                      │                                      │
│ follows    │                      │  2. Fallback: popular posts          │
│ posts      │                      │     for cold-start users             │
│ user_inter │                      └───────────────┬──────────────────────┘
└────────────┘                                      │
                                                    │ write
                                                    ▼
┌─────────┐   GET /explore     ┌───────────┐   ┌──────────┐
│ Client  │───────────────────►│  Explore  │──►│  Redis   │
│         │◄───────────────────│  Service  │   │          │
│         │   top 30 posts     │ (hydrate) │   │ explore: │
└─────────┘                    └───────────┘   │ {user_id}│
                                               └──────────┘
```

## 🧪 Simulating User Interaction and Re-ranking

One powerful concept: as the user interacts with Explore content,  
we can **update their recommendations in real-time** (not just every 30 minutes).

If User 1 likes a post from User 30 on Explore, we can immediately  
boost other posts from User 30 and similar content.

In [ ]:
def record_explore_interaction(user_id: int, post_id: int, interaction_type: str):
    """
    Record a user's interaction with an Explore post.
    This updates the recommendations in real-time.
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    # Save interaction to database
    cur.execute(
        """INSERT INTO user_interactions (user_id, post_id, interaction_type)
           VALUES (%s, %s, %s)""",
        (user_id, post_id, interaction_type)
    )

    # Get the author of the liked post
    cur.execute("SELECT author_id FROM posts WHERE id = %s", (post_id,))
    row = cur.fetchone()
    conn.commit()
    conn.close()

    if not row:
        return

    author_id = row[0]

    # Boost other posts from the same author in the Explore feed
    explore_key = f"explore:{user_id}"
    boost = {"like": 0.15, "save": 0.20, "comment": 0.10, "view": 0.02}
    boost_amount = boost.get(interaction_type, 0.05)

    # Find other posts by this author in the Explore set and boost them
    all_explore = r.zrevrange(explore_key, 0, -1, withscores=True)
    conn2 = get_db()
    cur2 = conn2.cursor()

    boosted = 0
    for pid_str, score in all_explore:
        cur2.execute("SELECT author_id FROM posts WHERE id = %s", (int(pid_str),))
        r2 = cur2.fetchone()
        if r2 and r2[0] == author_id:
            r.zincrby(explore_key, boost_amount, pid_str)
            boosted += 1

    conn2.close()
    print(f"👆 User {user_id} {interaction_type}d post #{post_id} (by user {author_id})")
    print(f"   Boosted {boosted} other posts from user {author_id} by +{boost_amount}")

# Simulate: User 1 likes a post from their Explore feed
if explore:
    first_post = explore[0]
    print("Before interaction:")
    print(f"   Post #{first_post['post_id']} score: {first_post['explore_score']:.4f}\n")

    record_explore_interaction(user_id=1, post_id=first_post["post_id"], interaction_type="like")

    print("\nAfter interaction — scores updated in real-time!")
    print("→ Open RedisInsight to see the updated scores in explore:1")
else:
    print("No explore posts to interact with.")

## 🧠 Key Takeaways

1. **Two-stage pipeline**: candidate generation (find posts) → ranking (score and sort)
2. **Collaborative filtering**: "users like you also liked..." — simple but effective
3. **Engagement scoring**: weight multiple signals (likes, recency, author quality, diversity)
4. **Cold start fallback**: popularity-based recommendations for new users
5. **Pre-compute + cache**: batch job generates recommendations, Redis serves them instantly
6. **Real-time updates**: boost scores based on user interactions within the session

### Interview Tips

- Describe the two-stage pipeline (candidate generation → ranking)
- Mention collaborative filtering as the core approach
- Address the **cold start problem** — what do you show new users?
- Talk about pre-computation vs real-time: batch jobs for base scores, real-time boosts for freshness
- Discuss **diversity** — don't show 10 posts from the same author
- At scale, mention that ML models (neural networks) replace the simple scoring function

### Lab Complete! 🎉

You've now built the core systems behind Instagram:
1. ✅ **Photo Upload Pipeline** — pre-signed URLs, thumbnails, CDN caching
2. ✅ **News Feed Generation** — fan-out on write, hybrid celebrity approach
3. ✅ **Stories** — Redis TTL for ephemeral content, view tracking
4. ✅ **Explore & Recommendations** — collaborative filtering, engagement scoring